# Calibración del rojo y el amarillo — revisión remota de soldaduras

La app muestra todo lo que supera el umbral del modelo (elegido para no perder defectos). Entre
eso, **rojo** quiere decir "el modelo suele acertar" y **amarillo punteado** "duda, conviene
mirarlo". Este notebook elige el corte entre los dos:

1. Baja los datasets y arma los **mismos conjuntos** que el entrenamiento (misma semilla).
2. Corre los modelos **ONNX de la app** sobre validación, con el mismo procesamiento que la web.
3. Para cada corte posible mide la **precisión** (qué parte de los recuadros marca un defecto
   real) y elige el corte más bajo con al menos 80 % de precisión.
4. Comprueba ese corte en **prueba** (imágenes que no se usaron para elegirlo).

**Antes de correr:** *Internet: On*, el secreto `ROBOFLOW_API_KEY` y como entradas (*Add Input →
Notebooks*) las salidas de los notebooks de radiografías y de cordones, que tienen los `.onnx`.
No hace falta GPU.

In [ ]:
!pip install -q roboflow imagehash onnxruntime
import os
os.makedirs('revision_pipes', exist_ok=True)
open('revision_pipes/__init__.py', 'w').close()

In [ ]:
%%writefile revision_pipes/annotations.py
"""Formato común de anotaciones (recuadros) para todos los datasets.

Cada dataset trae los recuadros en un formato distinto:
- YOLO: un .txt por imagen, con valores relativos (0 a 1) al tamaño de la imagen.
- COCO: un .json con [x, y, ancho, alto] en píxeles.
- Polígonos (SWRD): lista de puntos que rodean el defecto.

Este módulo los lleva todos a `Box`, en píxeles absolutos (x_min, y_min, x_max, y_max),
para que el resto del proyecto no dependa del formato de origen.
"""

from __future__ import annotations

from collections.abc import Iterable, Sequence
from dataclasses import dataclass

MIN_BOX_SIDE = 1.0


class AnnotationError(ValueError):
    """Una anotación no se puede interpretar."""


@dataclass(frozen=True)
class Box:
    x_min: float
    y_min: float
    x_max: float
    y_max: float
    label: str
    score: float | None = None  # confianza del modelo; None si es una anotación humana

    @property
    def width(self) -> float:
        return self.x_max - self.x_min

    @property
    def height(self) -> float:
        return self.y_max - self.y_min

    @property
    def area(self) -> float:
        return max(self.width, 0.0) * max(self.height, 0.0)

    def clip(self, image_width: int, image_height: int) -> Box:
        """Recorta el recuadro para que no se salga de la imagen."""
        return Box(
            x_min=min(max(self.x_min, 0.0), image_width),
            y_min=min(max(self.y_min, 0.0), image_height),
            x_max=min(max(self.x_max, 0.0), image_width),
            y_max=min(max(self.y_max, 0.0), image_height),
            label=self.label,
            score=self.score,
        )

    def is_valid(self) -> bool:
        return self.width >= MIN_BOX_SIDE and self.height >= MIN_BOX_SIDE


def from_yolo(
    text: str, image_width: int, image_height: int, class_names: Sequence[str]
) -> list[Box]:
    """Lee un .txt de YOLO: `clase x_centro y_centro ancho alto` (relativos, 0 a 1)."""
    boxes = []
    for line_number, line in enumerate(text.splitlines(), start=1):
        parts = line.split()
        if not parts:
            continue
        if len(parts) != 5:
            raise AnnotationError(f"Línea {line_number}: se esperaban 5 valores, hay {len(parts)}.")
        try:
            class_id = int(parts[0])
            xc, yc, w, h = (float(value) for value in parts[1:])
        except ValueError as exc:
            raise AnnotationError(f"Línea {line_number}: valores no numéricos.") from exc
        if not 0 <= class_id < len(class_names):
            raise AnnotationError(f"Línea {line_number}: clase {class_id} desconocida.")
        boxes.append(
            Box(
                x_min=(xc - w / 2) * image_width,
                y_min=(yc - h / 2) * image_height,
                x_max=(xc + w / 2) * image_width,
                y_max=(yc + h / 2) * image_height,
                label=class_names[class_id],
            )
        )
    return boxes


def from_coco(coco: dict) -> dict[str, list[Box]]:
    """Lee un diccionario COCO y devuelve los recuadros agrupados por nombre de archivo."""
    try:
        categories = {c["id"]: c["name"] for c in coco["categories"]}
        images = {img["id"]: img["file_name"] for img in coco["images"]}
        result: dict[str, list[Box]] = {name: [] for name in images.values()}
        for ann in coco["annotations"]:
            x, y, w, h = ann["bbox"]
            result[images[ann["image_id"]]].append(
                Box(x, y, x + w, y + h, categories[ann["category_id"]])
            )
    except (KeyError, TypeError, ValueError) as exc:
        raise AnnotationError(f"Archivo COCO mal formado: {exc}") from exc
    return result


def from_polygon(points: Iterable[Sequence[float]], label: str) -> Box:
    """Convierte un polígono (lista de puntos x, y) en el recuadro que lo contiene."""
    points = list(points)
    if len(points) < 3:
        raise AnnotationError("Un polígono necesita al menos 3 puntos.")
    xs = [float(p[0]) for p in points]
    ys = [float(p[1]) for p in points]
    return Box(min(xs), min(ys), max(xs), max(ys), label)


def clean(boxes: Iterable[Box], image_width: int, image_height: int) -> list[Box]:
    """Recorta los recuadros a la imagen y descarta los que quedan sin tamaño."""
    clipped = (box.clip(image_width, image_height) for box in boxes)
    return [box for box in clipped if box.is_valid()]


def iou(a: Box, b: Box) -> float:
    """Intersección sobre unión: cuánto se superponen dos recuadros (0 a 1)."""
    ix = max(0.0, min(a.x_max, b.x_max) - max(a.x_min, b.x_min))
    iy = max(0.0, min(a.y_max, b.y_max) - max(a.y_min, b.y_min))
    inter = ix * iy
    union = a.area + b.area - inter
    return inter / union if union > 0 else 0.0

In [ ]:
%%writefile revision_pipes/splits.py
"""Separación en entrenamiento, validación y prueba sin fuga de datos.

Si dos recortes salen de la misma placa radiográfica y uno cae en entrenamiento
y otro en prueba, el modelo "ya vio" esa placa y las métricas salen infladas.
Por eso se separa por grupo (la placa de origen), no por imagen suelta.
"""

from __future__ import annotations

import random
from collections.abc import Callable, Sequence
from typing import TypeVar

T = TypeVar("T")


def split_by_group(
    items: Sequence[T],
    group_of: Callable[[T], str],
    ratios: tuple[float, float, float] = (0.7, 0.15, 0.15),
    seed: int = 42,
) -> dict[str, list[T]]:
    """Reparte los grupos completos entre train, val y test según las proporciones.

    Es reproducible: con la misma semilla siempre da la misma separación.
    """
    if len(ratios) != 3 or any(r < 0 for r in ratios) or abs(sum(ratios) - 1.0) > 1e-6:
        raise ValueError("Las proporciones deben ser tres valores no negativos que sumen 1.")

    groups: dict[str, list[T]] = {}
    for item in items:
        groups.setdefault(group_of(item), []).append(item)

    names = sorted(groups)
    random.Random(seed).shuffle(names)

    n = len(names)
    n_train = round(n * ratios[0])
    n_val = round(n * ratios[1])
    assignment = {
        "train": names[:n_train],
        "val": names[n_train : n_train + n_val],
        "test": names[n_train + n_val :],
    }
    return {split: [item for g in gs for item in groups[g]] for split, gs in assignment.items()}

In [ ]:
%%writefile revision_pipes/taxonomy.py
"""Nombres comunes para los tipos de defecto en radiografías.

Cada dataset nombra los defectos a su manera ("Lack Of Penetration", "incomplete penetration",
"LOP", "Tungsten Inclusion" y "tungsten inclusion" en el mismo dataset...). Para entrenar un
solo modelo hace falta una lista única de clases, con nombres en español para la app.
"""

from __future__ import annotations

import re

# Nombre normalizado de entrada (minúsculas, sin signos) -> clase común
ALIASES = {
    "crack": "fisura",
    "cracks": "fisura",
    "porosity": "porosidad",
    "pore": "porosidad",
    "blowhole": "porosidad",
    "slag": "inclusion de escoria",
    "slag inclusion": "inclusion de escoria",
    "slag inclusions": "inclusion de escoria",
    "lack of fusion": "falta de fusion",
    "incomplete fusion": "falta de fusion",
    "lof": "falta de fusion",
    "lack of penetration": "falta de penetracion",
    "incomplete penetration": "falta de penetracion",
    "lop": "falta de penetracion",
    "non penetration": "falta de penetracion",
    "undercut": "socavadura",
    "undarcut": "socavadura",
    "burn through": "perforacion",
    "burnthrough": "perforacion",
    "tungsten inclusion": "inclusion de tungsteno",
    "excessive reinforcement": "sobremonta excesiva",
    "inadequate reinforcement": "sobremonta insuficiente",
    "root concavity": "concavidad de raiz",
    "good weld": "sin defecto",
    "no defect": "sin defecto",
}

COMMON_CLASSES = sorted(set(ALIASES.values()))


def _key(label: str) -> str:
    return re.sub(r"[^a-z]+", " ", label.lower()).strip()


def normalize_label(label: str) -> str:
    """Devuelve la clase común. Si el nombre no se conoce, devuelve 'otro: <original>'
    para que aparezca en el análisis y se pueda decidir qué hacer con él."""
    return ALIASES.get(_key(label), f"otro: {label.strip()}")


# Clases con las que se entrena el detector de radiografías (Etapa 2).
# Las que tienen pocos ejemplos se agrupan en "otro defecto": el modelo igual avisa
# que hay algo, y el inspector decide qué es.
TRAINING_CLASSES_RX = [
    "porosidad",
    "fisura",
    "inclusion de escoria",
    "falta de penetracion",
    "falta de fusion",
    "otro defecto",
]
TRAINING_CLASSES_CORDONES = ["Good Weld", "Bad Weld", "Defect"]


def to_training_class(common_label: str) -> str | None:
    """Pasa una clase común a una de las clases de entrenamiento de radiografías.

    Devuelve None para "sin defecto" (no es un recuadro a detectar).
    """
    if common_label == "sin defecto":
        return None
    if common_label in TRAINING_CLASSES_RX:
        return common_label
    return "otro defecto"

In [ ]:
%%writefile revision_pipes/visualize.py
"""Dibujo de recuadros sobre la imagen: rojo para defectos, verde para zonas sanas.

Es el componente visual del proyecto: subo una placa o una foto y veo
dónde está el problema y dónde está bien.
"""

from __future__ import annotations

from collections.abc import Iterable
from pathlib import Path

from PIL import Image, ImageDraw, ImageFont

from revision_pipes.annotations import Box

FONT = Path(__file__).parent / "assets" / "DejaVuSans.ttf"

RED = (220, 38, 38)
GREEN = (22, 163, 74)
AMBER = (249, 115, 22)  # naranja: cordón malo

# Etiquetas que se consideran "sanas" (se dibujan en verde). El resto va en rojo.
GOOD_LABELS = {"good weld", "good_weld", "ok", "sin defecto", "no defect", "pipe"}
# Etiquetas que indican un problema general (cordón malo), en ámbar.
WARNING_LABELS = {"bad weld", "bad_weld"}


def color_for(label: str) -> tuple[int, int, int]:
    key = label.strip().lower()
    if key in GOOD_LABELS:
        return GREEN
    if key in WARNING_LABELS:
        return AMBER
    return RED


def draw_boxes(
    image: Image.Image, boxes: Iterable[Box], show_numbers: bool = True
) -> Image.Image:
    """Devuelve una copia de la imagen con los recuadros numerados y coloreados."""
    canvas = image.convert("RGB").copy()
    draw = ImageDraw.Draw(canvas)
    thickness = max(2, round(min(canvas.size) / 200))
    for number, box in enumerate(boxes, start=1):
        color = color_for(box.label)
        draw.rectangle(
            [box.x_min, box.y_min, box.x_max, box.y_max], outline=color, width=thickness
        )
        if show_numbers:
            tag = f"{number} {box.label}"
            if box.score is not None:
                tag += f" {box.score:.0%}"
            left, top, right, bottom = draw.textbbox((box.x_min, box.y_min), tag)
            draw.rectangle([left - 2, top - 2, right + 2, bottom + 2], fill=color)
            draw.text((box.x_min, box.y_min), tag, fill=(255, 255, 255))
    return canvas


def summarize(boxes: Iterable[Box]) -> dict[str, int]:
    """Cuenta los recuadros por etiqueta (sirve también para contar caños)."""
    counts: dict[str, int] = {}
    for box in boxes:
        counts[box.label] = counts.get(box.label, 0) + 1
    return dict(sorted(counts.items()))


YELLOW = (234, 179, 8)
LEVEL_STYLE = {
    # nivel: (color, prefijo del texto, grosor relativo)
    "defecto": (RED, "", 1.0),
    "revisar": (YELLOW, "? ", 0.6),
    "bien": (GREEN, "", 1.0),
}


def draw_findings(image: Image.Image, findings) -> Image.Image:
    """Dibuja hallazgos clasificados: rojo seguro, amarillo "revisar", verde sano.

    El cordón malo seguro se dibuja en ámbar para distinguirlo de un defecto puntual.
    El color nunca va solo: cada recuadro lleva número, nombre y porcentaje.
    """
    canvas = image.convert("RGB").copy()
    draw = ImageDraw.Draw(canvas)
    base = max(2, round(min(canvas.size) / 200))
    # Fuente con tildes y tamaño proporcional a la imagen (la fuente básica no tiene "ó").
    font = ImageFont.truetype(str(FONT), max(11, round(min(canvas.size) / 40)))
    # Se dibujan del menos importante al más importante, para que el rojo quede arriba.
    for finding in sorted(findings, key=lambda f: -f.number):
        color, prefix, weight = LEVEL_STYLE[finding.level]
        if finding.level == "defecto" and finding.box.label.strip().lower() in WARNING_LABELS:
            color = AMBER
        box = finding.box
        draw.rectangle(
            [box.x_min, box.y_min, box.x_max, box.y_max],
            outline=color,
            width=max(1, round(base * weight)),
        )
        tag = f"{prefix}{finding.number} {finding.name}"
        if box.score is not None:
            tag += f" {box.score:.0%}"
        left, top, right, bottom = draw.textbbox((box.x_min, box.y_min), tag, font=font)
        draw.rectangle([left - 2, top - 2, right + 2, bottom + 2], fill=color)
        ink = (0, 0, 0) if color == YELLOW else (255, 255, 255)
        draw.text((box.x_min, box.y_min), tag, fill=ink, font=font)
    return canvas

In [ ]:
%%writefile revision_pipes/metrics.py
"""Métricas pensadas para el inspector.

- Recall por placa: de las placas que tienen al menos un defecto, ¿cuántas marcó el modelo?
  Es lo que más importa: una placa con defecto que pasa sin marcar es el peor error.
- Recall y precisión por clase, emparejando recuadros con IoU ≥ 0,5.
- Elección del umbral de confianza: el más alto que todavía cumple el recall pedido.
"""

from __future__ import annotations

from collections.abc import Sequence
from dataclasses import dataclass

from revision_pipes.annotations import Box, iou


@dataclass(frozen=True)
class ClassMetrics:
    label: str
    true_positives: int
    false_positives: int
    false_negatives: int

    @property
    def recall(self) -> float:
        total = self.true_positives + self.false_negatives
        return self.true_positives / total if total else 0.0

    @property
    def precision(self) -> float:
        total = self.true_positives + self.false_positives
        return self.true_positives / total if total else 0.0


def above(boxes: Sequence[Box], threshold: float) -> list[Box]:
    return [b for b in boxes if (b.score if b.score is not None else 1.0) >= threshold]


def image_recall(
    predictions: Sequence[Sequence[Box]], truths: Sequence[Sequence[Box]], threshold: float
) -> float:
    """Proporción de imágenes con defectos reales donde el modelo marcó al menos uno."""
    with_defects = [i for i, t in enumerate(truths) if t]
    if not with_defects:
        return 0.0
    hits = sum(1 for i in with_defects if above(predictions[i], threshold))
    return hits / len(with_defects)


def choose_threshold(
    predictions: Sequence[Sequence[Box]],
    truths: Sequence[Sequence[Box]],
    target_recall: float = 0.95,
    candidates: Sequence[float] | None = None,
) -> float:
    """El umbral más alto (menos falsas alarmas) que cumple el recall por placa pedido.

    Si ninguno lo cumple, devuelve el más bajo de los candidatos.
    """
    if candidates is None:
        candidates = [round(0.05 * i, 2) for i in range(1, 20)]
    for threshold in sorted(candidates, reverse=True):
        if image_recall(predictions, truths, threshold) >= target_recall:
            return threshold
    return min(candidates)


def per_class(
    predictions: Sequence[Sequence[Box]],
    truths: Sequence[Sequence[Box]],
    threshold: float,
    iou_threshold: float = 0.5,
) -> dict[str, ClassMetrics]:
    """Empareja recuadros predichos y reales de la misma clase (cada real se usa una vez)."""
    counts: dict[str, list[int]] = {}
    for preds, gts in zip(predictions, truths, strict=True):
        preds = sorted(above(preds, threshold), key=lambda b: -(b.score or 1.0))
        used = [False] * len(gts)
        for p in preds:
            best, best_j = 0.0, -1
            for j, g in enumerate(gts):
                if not used[j] and g.label == p.label:
                    overlap = iou(p, g)
                    if overlap > best:
                        best, best_j = overlap, j
            c = counts.setdefault(p.label, [0, 0, 0])
            if best_j >= 0 and best >= iou_threshold:
                used[best_j] = True
                c[0] += 1
            else:
                c[1] += 1
        for j, g in enumerate(gts):
            if not used[j]:
                counts.setdefault(g.label, [0, 0, 0])[2] += 1
    return {k: ClassMetrics(k, *v) for k, v in sorted(counts.items())}

In [ ]:
%%writefile revision_pipes/tiling.py
"""Ventanas superpuestas para placas alargadas.

Una placa completa puede medir 3000 × 600 píxeles. Si se achica entera al cuadrado de
640 × 640 que recibe el modelo, un poro de pocos píxeles desaparece. En cambio se corta
en ventanas cuadradas (del lado más corto) que se pisan un 25 %, se detecta en cada una
y los recuadros se devuelven a coordenadas de la placa, uniendo los duplicados.
"""

from __future__ import annotations

from collections.abc import Iterable, Sequence

from revision_pipes.annotations import Box, iou

Window = tuple[int, int, int, int]  # x_min, y_min, x_max, y_max


def windows(width: int, height: int, overlap: float = 0.25, min_ratio: float = 1.5) -> list[Window]:
    """Ventanas cuadradas que cubren toda la imagen.

    Si la imagen es casi cuadrada (lado largo / lado corto < min_ratio), devuelve una sola
    ventana con la imagen entera.
    """
    if not 0.0 <= overlap < 1.0:
        raise ValueError("La superposición debe estar entre 0 y 1 (sin incluir 1).")
    long_side, short_side = max(width, height), min(width, height)
    if short_side <= 0:
        raise ValueError("La imagen no tiene tamaño.")
    if long_side / short_side < min_ratio:
        return [(0, 0, width, height)]
    step = max(1, int(short_side * (1 - overlap)))
    starts = list(range(0, long_side - short_side, step)) + [long_side - short_side]
    starts = sorted(set(starts))
    if width >= height:
        return [(s, 0, s + short_side, height) for s in starts]
    return [(0, s, width, s + short_side) for s in starts]


def boxes_in_window(boxes: Iterable[Box], window: Window, min_visible: float = 0.4) -> list[Box]:
    """Recuadros de la placa expresados dentro de la ventana.

    Un recuadro cortado por el borde se conserva solo si queda visible al menos
    `min_visible` de su área.
    """
    x0, y0, x1, y1 = window
    result = []
    for b in boxes:
        clipped = Box(
            max(b.x_min, x0), max(b.y_min, y0), min(b.x_max, x1), min(b.y_max, y1), b.label, b.score
        )
        if clipped.width <= 0 or clipped.height <= 0 or b.area <= 0:
            continue
        if clipped.area / b.area < min_visible:
            continue
        result.append(
            Box(clipped.x_min - x0, clipped.y_min - y0, clipped.x_max - x0, clipped.y_max - y0,
                b.label, b.score)
        )
    return result


def to_plate(boxes: Iterable[Box], window: Window) -> list[Box]:
    """Pasa recuadros de coordenadas de la ventana a coordenadas de la placa."""
    x0, y0 = window[0], window[1]
    return [Box(b.x_min + x0, b.y_min + y0, b.x_max + x0, b.y_max + y0, b.label, b.score)
            for b in boxes]


def merge(boxes: Sequence[Box], iou_threshold: float = 0.5) -> list[Box]:
    """Une duplicados de la misma clase (de ventanas que se pisan): queda el de más confianza."""
    ordered = sorted(boxes, key=lambda b: -(b.score if b.score is not None else 1.0))
    kept: list[Box] = []
    for b in ordered:
        if all(k.label != b.label or iou(k, b) < iou_threshold for k in kept):
            kept.append(b)
    return kept

In [ ]:
%%writefile revision_pipes/inference.py
"""Inferencia con el modelo exportado a ONNX (RT-DETR).

El modelo recibe una imagen RGB de 640x640 con valores entre 0 y 1 y devuelve, para cada
una de sus 300 "consultas", un puntaje por clase (`logits`) y un recuadro relativo en
formato centro-ancho-alto (`pred_boxes`). Acá se convierte eso en recuadros `Box`
en píxeles de la imagen original.

La sesión de ONNX se recibe desde afuera para poder testear sin el modelo real.
"""

from __future__ import annotations

from collections.abc import Sequence
from typing import Protocol

import numpy as np
from PIL import Image

from revision_pipes.annotations import Box
from revision_pipes.tiling import merge, to_plate, windows

INPUT_SIZE = 640


class Session(Protocol):
    def run(self, output_names, input_feed: dict) -> list: ...


def prepare(image: Image.Image, size: int = INPUT_SIZE) -> np.ndarray:
    """Imagen → tensor (1, 3, size, size) float32 en [0, 1], como en el entrenamiento."""
    rgb = image.convert("RGB").resize((size, size), Image.Resampling.BILINEAR)
    array = np.asarray(rgb, dtype=np.float32) / 255.0
    return array.transpose(2, 0, 1)[None]


def decode(
    logits: np.ndarray,
    pred_boxes: np.ndarray,
    image_size: tuple[int, int],
    labels: Sequence[str],
    threshold: float,
) -> list[Box]:
    """Convierte la salida cruda del modelo en recuadros por encima del umbral."""
    if logits.ndim == 3:
        logits, pred_boxes = logits[0], pred_boxes[0]
    if logits.shape[-1] != len(labels):
        raise ValueError(f"El modelo tiene {logits.shape[-1]} clases y se pasaron {len(labels)}.")
    scores = 1.0 / (1.0 + np.exp(-logits))  # RT-DETR usa sigmoide por clase
    best_class = scores.argmax(axis=1)
    best_score = scores.max(axis=1)
    width, height = image_size
    boxes = []
    for query in np.argsort(-best_score, kind="stable"):
        score = float(best_score[query])
        if score < threshold:
            break
        cx, cy, w, h = pred_boxes[query]
        boxes.append(
            Box(
                x_min=float((cx - w / 2) * width),
                y_min=float((cy - h / 2) * height),
                x_max=float((cx + w / 2) * width),
                y_max=float((cy + h / 2) * height),
                label=labels[int(best_class[query])],
                score=score,
            ).clip(width, height)
        )
    return [b for b in boxes if b.is_valid()]


class Detector:
    """Envuelve una sesión de ONNX Runtime con las etiquetas y el umbral elegidos."""

    def __init__(self, session: Session, labels: Sequence[str], threshold: float):
        if not 0.0 <= threshold <= 1.0:
            raise ValueError("El umbral debe estar entre 0 y 1.")
        self.session = session
        self.labels = list(labels)
        self.threshold = threshold

    @classmethod
    def from_file(cls, model_path: str, labels: Sequence[str], threshold: float) -> Detector:
        import onnxruntime as ort  # opcional: solo hace falta para usar el modelo real

        session = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])
        return cls(session, labels, threshold)

    def predict(self, image: Image.Image) -> list[Box]:
        """Detecta en la imagen. Si es alargada, la corta en ventanas superpuestas y une."""
        found = []
        for window in windows(*image.size):
            crop = image.crop(window)
            logits, pred_boxes = self.session.run(None, {"pixel_values": prepare(crop)})[:2]
            found += to_plate(decode(logits, pred_boxes, crop.size, self.labels, self.threshold),
                              window)
        return merge(found)

In [ ]:
%%writefile revision_pipes/review.py
"""Lectura de los hallazgos para el inspector: qué es defecto, qué conviene revisar, qué está bien.

El modelo devuelve recuadros con un puntaje. Para no llenar la pantalla de rojo, cada hallazgo
se clasifica en un nivel:

- "defecto": un problema y el modelo está seguro (puntaje >= umbral alto).
- "revisar": un problema posible, pero con poca confianza. La IA lo señala; el inspector decide.
- "bien": zona sana (por ejemplo, "cordón bueno" en las fotos).

El umbral del modelo (el más bajo) se eligió para no perder defectos; el umbral alto solo cambia
cómo se muestra el hallazgo, nunca lo oculta.
"""

from __future__ import annotations

from collections.abc import Iterable, Sequence
from dataclasses import dataclass

from revision_pipes.annotations import Box

DEFECTO = "defecto"
REVISAR = "revisar"
BIEN = "bien"

# Nombres para mostrar en la app. Las clases de radiografías ya vienen en español.
DISPLAY_NAMES = {
    "Good Weld": "Cordón bueno",
    "Bad Weld": "Cordón malo",
    "Defect": "Defecto puntual",
    "inclusion de escoria": "Inclusión de escoria",
    "falta de penetracion": "Falta de penetración",
    "falta de fusion": "Falta de fusión",
}

GOOD_LABELS = {"Good Weld"}


def display_name(label: str) -> str:
    return DISPLAY_NAMES.get(label, label[:1].upper() + label[1:])


@dataclass(frozen=True)
class Finding:
    number: int
    box: Box
    level: str

    @property
    def name(self) -> str:
        return display_name(self.box.label)


def _area(b: Box) -> float:
    return max((b.x_max - b.x_min) * (b.y_max - b.y_min), 1e-9)


def _overlap(a: Box, b: Box) -> float:
    """Intersección sobre el área del recuadro más chico (1 = uno está entero dentro del otro)."""
    w = max(0.0, min(a.x_max, b.x_max) - max(a.x_min, b.x_min))
    h = max(0.0, min(a.y_max, b.y_max) - max(a.y_min, b.y_min))
    return w * h / min(_area(a), _area(b))


def drop_nested(boxes: Sequence[Box], min_inside: float = 0.8) -> list[Box]:
    """Quita recuadros encimados de la misma clase: si uno queda casi entero dentro del otro
    (en cualquier sentido), se conserva el más seguro.

    Solo afecta lo que se muestra: el modelo a veces marca el mismo cordón entero y además
    medio cordón, y en pantalla eso son tres recuadros encimados para un único hallazgo.
    """
    kept: list[Box] = []
    for box in sorted(boxes, key=lambda b: -(b.score or 0.0)):
        inside = any(k.label == box.label and _overlap(k, box) >= min_inside for k in kept)
        if not inside:
            kept.append(box)
    return kept


def classify(boxes: Sequence[Box], high_threshold: float) -> list[Finding]:
    """Ordena los hallazgos (primero los defectos seguros) y les asigna un nivel."""
    if not 0.0 <= high_threshold <= 1.0:
        raise ValueError("El umbral alto debe estar entre 0 y 1.")

    def level(box: Box) -> str:
        if box.label in GOOD_LABELS:
            return BIEN
        return DEFECTO if (box.score or 0.0) >= high_threshold else REVISAR

    order = {DEFECTO: 0, REVISAR: 1, BIEN: 2}
    ranked = sorted(boxes, key=lambda b: (order[level(b)], -(b.score or 0.0)))
    return [Finding(i, b, level(b)) for i, b in enumerate(ranked, start=1)]


def verdict(findings: Iterable[Finding]) -> str:
    """Resumen en una línea para la pantalla. Es una sugerencia: decide el inspector."""
    levels = [f.level for f in findings]
    n_def, n_rev = levels.count(DEFECTO), levels.count(REVISAR)
    if n_def:
        extra = f" y {n_rev} para revisar" if n_rev else ""
        return f"{n_def} posible{'s' if n_def > 1 else ''} defecto{'s' if n_def > 1 else ''}{extra}"
    if n_rev:
        return f"Sin defectos claros · {n_rev} zona{'s' if n_rev > 1 else ''} para revisar"
    return "Sin hallazgos"

## Radiografias

In [ ]:
import os

MODULO = "radiografias"      # "radiografias" o "cordones"
EPOCHS = 25                   # el primer modelo llegó a su mejor valor en la época 13
IMG_SIZE = 640
BATCH = 8
LR = 1e-4                     # cabeza del detector
LR_BACKBONE = 1e-5            # el backbone preentrenado se ajusta más suave
SEED = 42                     # la misma de la Etapa 1: mismos conjuntos
TARGET_RECALL = 0.97          # en validación se apunta más alto: margen para que en prueba quede ≥ 95 %
OVERLAP = 0.25                # superposición de las ventanas en placas alargadas
MIN_RATIO = 1.5               # desde qué proporción (largo / corto) se corta en ventanas
VERSION = "v2-ventanas"
TIME_LIMIT_H = 3.0            # corte de seguridad para no agotar la cuota de GPU
MODEL_ID = "PekingU/rtdetr_r18vd_coco_o365"
DRY_RUN = os.environ.get("DRY_RUN") == "1"  # prueba rápida con un modelo chico, sin GPU

SOURCES = {
    "rx_weld": ("kunkun-vhmx2", "weld-2z7bw", 1, "radiografias"),
    "rx_weld_data": ("vayun-goel-jlreu", "weld-data-4ln0h", None, "radiografias"),
    "cordones_ccby": ("computer-vision-5vq1v", "welding_defects-2tvbr", 1, "cordones"),
    "cordones_dominio_publico": ("computer-vision-pl2fz", "welding-defects-detection-ru1eb", None, "cordones"),
}
SOURCES = {k: v for k, v in SOURCES.items() if v[3] == MODULO}

In [ ]:
import json, math, random, time
from collections import Counter
from pathlib import Path

import imagehash
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image, ImageEnhance

from revision_pipes.annotations import Box, clean, from_coco, iou
from revision_pipes.inference import decode
from revision_pipes.metrics import choose_threshold, image_recall, per_class
from revision_pipes.splits import split_by_group
from revision_pipes.tiling import boxes_in_window, merge, to_plate, windows
from revision_pipes.taxonomy import (
    TRAINING_CLASSES_CORDONES, TRAINING_CLASSES_RX, normalize_label, to_training_class,
)
from revision_pipes.visualize import draw_boxes, summarize

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
WORK = Path("/kaggle/working")
DATA = Path("/tmp/data")
DATA.mkdir(parents=True, exist_ok=True)
LABELS = TRAINING_CLASSES_RX if MODULO == "radiografias" else TRAINING_CLASSES_CORDONES
# Clases que cuentan como "hay un problema" para el recall por imagen y las falsas alarmas.
# En cordones, "Good Weld" es un recuadro verde: no es un defecto.
DEFECT_LABELS = set(LABELS) - {"Good Weld"}


def defects(boxes):
    return [b for b in boxes if b.label in DEFECT_LABELS]

In [ ]:
if not DRY_RUN:
    from kaggle_secrets import UserSecretsClient
    from roboflow import Roboflow

    rf = Roboflow(api_key=UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))
    for name, (workspace, project_id, version, kind) in SOURCES.items():
        project = rf.workspace(workspace).project(project_id)
        if version is None:
            try:
                version = max(int(str(v.version).split("/")[-1]) for v in project.versions())
            except Exception:
                version = 1
        print(f"{name} → versión {version}")
        project.version(version).download("coco", location=str(DATA / name))

In [ ]:
def load_coco_folder(folder: Path) -> dict:
    result = {}
    for ann_file in folder.rglob("_annotations.coco.json"):
        for file_name, boxes in from_coco(json.loads(ann_file.read_text())).items():
            image_path = ann_file.parent / file_name
            if not image_path.exists():
                continue
            if MODULO == "radiografias":
                mapped = []
                for b in boxes:
                    label = to_training_class(normalize_label(b.label))
                    if label:
                        mapped.append(Box(b.x_min, b.y_min, b.x_max, b.y_max, label))
                boxes = mapped
            boxes = [b for b in boxes if b.label in LABELS]
            with Image.open(image_path) as im:
                result[image_path] = clean(boxes, *im.size)
    return result


def original_name(path: Path) -> str:
    return path.name.split(".rf.")[0]


def group_images(paths: list) -> dict:
    """Mismo agrupamiento que la Etapa 1: copias aumentadas y fotos casi idénticas juntas."""
    parent = {}

    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        parent[find(a)] = find(b)

    for path in paths:
        union(str(path), "orig:" + original_name(path))
    bits = []
    for path in paths:
        with Image.open(path) as im:
            bits.append(imagehash.phash(im.convert("L")).hash.flatten())
    bits = np.array(bits, dtype=bool)
    for i in range(len(paths)):
        distances = np.count_nonzero(bits[i + 1:] != bits[i], axis=1)
        for j in np.nonzero(distances <= 4)[0]:
            union(str(paths[i]), str(paths[i + 1 + j]))
    return {p: find(str(p)) for p in paths}


items = {}
for name in SOURCES:
    items.update(load_coco_folder(DATA / name))
groups = group_images(sorted(items))
splits = split_by_group(sorted(items), lambda p: groups[p], seed=SEED)
# En validación y prueba se deja UNA imagen por grupo: las copias aumentadas (rotadas,
# espejadas) de la misma foto contarían el mismo caso varias veces.
for split in ("val", "test"):
    seen, unique = set(), []
    for p in splits[split]:
        if groups[p] not in seen:
            seen.add(groups[p])
            unique.append(p)
    splits[split] = unique
for split, paths in splits.items():
    counts = Counter(b.label for p in paths for b in items[p])
    print(f"{split:5s}: {len(paths)} imágenes · {dict(counts.most_common())}")

# Placas alargadas → ventanas cuadradas superpuestas. Todas las ventanas de una placa quedan
# en el mismo conjunto que la placa (sin fuga).
sizes = {}
for p in items:
    with Image.open(p) as im:
        sizes[p] = im.size
plate_windows = {p: windows(*sizes[p], overlap=OVERLAP, min_ratio=MIN_RATIO) for p in items}
samples = {
    split: [(p, w, boxes_in_window(items[p], w)) for p in paths for w in plate_windows[p]]
    for split, paths in splits.items()
}
for split, ss in samples.items():
    healthy = sum(1 for _, _, b in ss if not b)
    print(f"{split:5s}: {len(ss)} ventanas de entrenamiento · {healthy} sin defectos (zonas sanas)")

In [ ]:
import zipfile
from revision_pipes.inference import Detector
from revision_pipes.review import drop_nested

TARGET_PRECISION = 0.80     # en rojo, al menos 8 de cada 10 recuadros tienen que ser defectos reales
MIN_RED = 15                # cantidad mínima de recuadros en rojo para confiar en la precisión
IOU_MATCH = 0.5
UMBRAL_MODELO = {"radiografias": 0.5, "cordones": 0.15}[MODULO]


def find_model(name: str) -> Path:
    hits = sorted(Path("/kaggle/input").rglob(name))
    if hits:
        return hits[0]
    for z in sorted(Path("/kaggle/input").rglob("*.zip")):
        with zipfile.ZipFile(z) as zf:
            inside = [n for n in zf.namelist() if n.endswith(name)]
            if inside:
                zf.extract(inside[0], "/tmp/modelos")
                return Path("/tmp/modelos") / inside[0]
    raise FileNotFoundError(f"No encuentro {name}: agregá la salida del notebook como entrada.")


onnx_path = find_model(f"modelo_{MODULO}.onnx")
print("Modelo:", onnx_path)
detector = Detector.from_file(str(onnx_path), LABELS, UMBRAL_MODELO)


def run(split: str) -> dict:
    out = {}
    for i, p in enumerate(splits[split]):
        with Image.open(p) as im:
            # igual que la app: ventanas, unión de duplicados y sin recuadros encimados
            out[p] = drop_nested(detector.predict(im.convert("RGB")))
        if i % 100 == 0:
            print(f"  {split}: {i}/{len(splits[split])}")
    return out


def scored(preds: dict) -> list:
    """(puntaje, acierta con cualquier defecto, acierta con el mismo tipo) por recuadro de defecto."""
    rows = []
    for p, boxes in preds.items():
        truth = defects(items[p])
        for b in defects(boxes):
            any_hit = any(iou(b, t) >= IOU_MATCH for t in truth)
            same_hit = any(iou(b, t) >= IOU_MATCH and t.label == b.label for t in truth)
            rows.append((b.score, any_hit, same_hit))
    return sorted(rows, reverse=True)


def curve(rows: list) -> list:
    points, hits, same = [], 0, 0
    total_hits = sum(r[1] for r in rows) or 1
    for n, (score, any_hit, same_hit) in enumerate(rows, start=1):
        hits += any_hit
        same += same_hit
        points.append({"umbral": round(score, 4), "rojos": n, "precision": hits / n,
                       "precision_mismo_tipo": same / n, "aciertos_en_rojo": hits / total_hits})
    return points


def summary(rows: list, threshold: float) -> dict:
    red = [r for r in rows if r[0] >= threshold]
    yellow = [r for r in rows if r[0] < threshold]
    pct = lambda xs, k: round(sum(x[k] for x in xs) / len(xs), 3) if xs else None
    return {"rojos": len(red), "precision_rojos": pct(red, 1), "precision_rojos_mismo_tipo": pct(red, 2),
            "amarillos": len(yellow), "precision_amarillos": pct(yellow, 1),
            "defectos_reales_en_rojo": round(sum(r[1] for r in red) / max(1, sum(r[1] for r in rows)), 3)}


print("Validación…")
val_rows = scored(run("val"))
points = curve(val_rows)
ok = [pt for pt in points if pt["rojos"] >= MIN_RED and pt["precision"] >= TARGET_PRECISION]
# El corte más bajo que sigue cumpliendo (el último en orden de puntaje descendente)
chosen = ok[-1]["umbral"] if ok else 1.0
chosen = max(chosen, UMBRAL_MODELO)
print(f"Umbral alto elegido: {chosen} ({'cumple' if ok else 'ningún corte llega al 80 %: todo amarillo'})")

print("Prueba…")
test_rows = scored(run("test"))
result = {
    "modulo": MODULO,
    "umbral_modelo": UMBRAL_MODELO,
    "umbral_alto": chosen,
    "objetivo_precision": TARGET_PRECISION,
    "validacion": summary(val_rows, chosen),
    "prueba": summary(test_rows, chosen),
    "curva_validacion": points[::max(1, len(points) // 60)],
}
print(json.dumps({k: v for k, v in result.items() if k != "curva_validacion"}, indent=2, ensure_ascii=False))
(WORK / f"calibracion_{MODULO}.json").write_text(json.dumps(result, indent=2, ensure_ascii=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([pt["umbral"] for pt in points], [pt["precision"] for pt in points], label="precisión de los rojos")
ax.plot([pt["umbral"] for pt in points], [pt["aciertos_en_rojo"] for pt in points],
        label="defectos reales que quedan en rojo")
ax.axvline(chosen, color="red", ls="--", label=f"corte elegido {chosen:.2f}")
ax.axhline(TARGET_PRECISION, color="gray", ls=":")
ax.set_xlabel("puntaje del modelo"); ax.set_ylim(0, 1.02); ax.legend(); ax.set_title(f"Calibración · {MODULO}")
plt.tight_layout(); plt.savefig(WORK / f"calibracion_{MODULO}.png", dpi=120); plt.show()

## Cordones

In [ ]:
import os

MODULO = "cordones"      # "radiografias" o "cordones"
EPOCHS = 25                   # el primer modelo llegó a su mejor valor en la época 13
IMG_SIZE = 640
BATCH = 8
LR = 1e-4                     # cabeza del detector
LR_BACKBONE = 1e-5            # el backbone preentrenado se ajusta más suave
SEED = 42                     # la misma de la Etapa 1: mismos conjuntos
TARGET_RECALL = 0.97          # en validación se apunta más alto: margen para que en prueba quede ≥ 95 %
OVERLAP = 0.25                # superposición de las ventanas en placas alargadas
MIN_RATIO = 1.5               # desde qué proporción (largo / corto) se corta en ventanas
VERSION = "v2-ventanas"
TIME_LIMIT_H = 3.0            # corte de seguridad para no agotar la cuota de GPU
MODEL_ID = "PekingU/rtdetr_r18vd_coco_o365"
DRY_RUN = os.environ.get("DRY_RUN") == "1"  # prueba rápida con un modelo chico, sin GPU

SOURCES = {
    "rx_weld": ("kunkun-vhmx2", "weld-2z7bw", 1, "radiografias"),
    "rx_weld_data": ("vayun-goel-jlreu", "weld-data-4ln0h", None, "radiografias"),
    "cordones_ccby": ("computer-vision-5vq1v", "welding_defects-2tvbr", 1, "cordones"),
    "cordones_dominio_publico": ("computer-vision-pl2fz", "welding-defects-detection-ru1eb", None, "cordones"),
}
SOURCES = {k: v for k, v in SOURCES.items() if v[3] == MODULO}

In [ ]:
import json, math, random, time
from collections import Counter
from pathlib import Path

import imagehash
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image, ImageEnhance

from revision_pipes.annotations import Box, clean, from_coco, iou
from revision_pipes.inference import decode
from revision_pipes.metrics import choose_threshold, image_recall, per_class
from revision_pipes.splits import split_by_group
from revision_pipes.tiling import boxes_in_window, merge, to_plate, windows
from revision_pipes.taxonomy import (
    TRAINING_CLASSES_CORDONES, TRAINING_CLASSES_RX, normalize_label, to_training_class,
)
from revision_pipes.visualize import draw_boxes, summarize

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
WORK = Path("/kaggle/working")
DATA = Path("/tmp/data")
DATA.mkdir(parents=True, exist_ok=True)
LABELS = TRAINING_CLASSES_RX if MODULO == "radiografias" else TRAINING_CLASSES_CORDONES
# Clases que cuentan como "hay un problema" para el recall por imagen y las falsas alarmas.
# En cordones, "Good Weld" es un recuadro verde: no es un defecto.
DEFECT_LABELS = set(LABELS) - {"Good Weld"}


def defects(boxes):
    return [b for b in boxes if b.label in DEFECT_LABELS]

In [ ]:
if not DRY_RUN:
    from kaggle_secrets import UserSecretsClient
    from roboflow import Roboflow

    rf = Roboflow(api_key=UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))
    for name, (workspace, project_id, version, kind) in SOURCES.items():
        project = rf.workspace(workspace).project(project_id)
        if version is None:
            try:
                version = max(int(str(v.version).split("/")[-1]) for v in project.versions())
            except Exception:
                version = 1
        print(f"{name} → versión {version}")
        project.version(version).download("coco", location=str(DATA / name))

In [ ]:
def load_coco_folder(folder: Path) -> dict:
    result = {}
    for ann_file in folder.rglob("_annotations.coco.json"):
        for file_name, boxes in from_coco(json.loads(ann_file.read_text())).items():
            image_path = ann_file.parent / file_name
            if not image_path.exists():
                continue
            if MODULO == "radiografias":
                mapped = []
                for b in boxes:
                    label = to_training_class(normalize_label(b.label))
                    if label:
                        mapped.append(Box(b.x_min, b.y_min, b.x_max, b.y_max, label))
                boxes = mapped
            boxes = [b for b in boxes if b.label in LABELS]
            with Image.open(image_path) as im:
                result[image_path] = clean(boxes, *im.size)
    return result


def original_name(path: Path) -> str:
    return path.name.split(".rf.")[0]


def group_images(paths: list) -> dict:
    """Mismo agrupamiento que la Etapa 1: copias aumentadas y fotos casi idénticas juntas."""
    parent = {}

    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        parent[find(a)] = find(b)

    for path in paths:
        union(str(path), "orig:" + original_name(path))
    bits = []
    for path in paths:
        with Image.open(path) as im:
            bits.append(imagehash.phash(im.convert("L")).hash.flatten())
    bits = np.array(bits, dtype=bool)
    for i in range(len(paths)):
        distances = np.count_nonzero(bits[i + 1:] != bits[i], axis=1)
        for j in np.nonzero(distances <= 4)[0]:
            union(str(paths[i]), str(paths[i + 1 + j]))
    return {p: find(str(p)) for p in paths}


items = {}
for name in SOURCES:
    items.update(load_coco_folder(DATA / name))
groups = group_images(sorted(items))
splits = split_by_group(sorted(items), lambda p: groups[p], seed=SEED)
# En validación y prueba se deja UNA imagen por grupo: las copias aumentadas (rotadas,
# espejadas) de la misma foto contarían el mismo caso varias veces.
for split in ("val", "test"):
    seen, unique = set(), []
    for p in splits[split]:
        if groups[p] not in seen:
            seen.add(groups[p])
            unique.append(p)
    splits[split] = unique
for split, paths in splits.items():
    counts = Counter(b.label for p in paths for b in items[p])
    print(f"{split:5s}: {len(paths)} imágenes · {dict(counts.most_common())}")

# Placas alargadas → ventanas cuadradas superpuestas. Todas las ventanas de una placa quedan
# en el mismo conjunto que la placa (sin fuga).
sizes = {}
for p in items:
    with Image.open(p) as im:
        sizes[p] = im.size
plate_windows = {p: windows(*sizes[p], overlap=OVERLAP, min_ratio=MIN_RATIO) for p in items}
samples = {
    split: [(p, w, boxes_in_window(items[p], w)) for p in paths for w in plate_windows[p]]
    for split, paths in splits.items()
}
for split, ss in samples.items():
    healthy = sum(1 for _, _, b in ss if not b)
    print(f"{split:5s}: {len(ss)} ventanas de entrenamiento · {healthy} sin defectos (zonas sanas)")

In [ ]:
import zipfile
from revision_pipes.inference import Detector
from revision_pipes.review import drop_nested

TARGET_PRECISION = 0.80     # en rojo, al menos 8 de cada 10 recuadros tienen que ser defectos reales
MIN_RED = 15                # cantidad mínima de recuadros en rojo para confiar en la precisión
IOU_MATCH = 0.5
UMBRAL_MODELO = {"radiografias": 0.5, "cordones": 0.15}[MODULO]


def find_model(name: str) -> Path:
    hits = sorted(Path("/kaggle/input").rglob(name))
    if hits:
        return hits[0]
    for z in sorted(Path("/kaggle/input").rglob("*.zip")):
        with zipfile.ZipFile(z) as zf:
            inside = [n for n in zf.namelist() if n.endswith(name)]
            if inside:
                zf.extract(inside[0], "/tmp/modelos")
                return Path("/tmp/modelos") / inside[0]
    raise FileNotFoundError(f"No encuentro {name}: agregá la salida del notebook como entrada.")


onnx_path = find_model(f"modelo_{MODULO}.onnx")
print("Modelo:", onnx_path)
detector = Detector.from_file(str(onnx_path), LABELS, UMBRAL_MODELO)


def run(split: str) -> dict:
    out = {}
    for i, p in enumerate(splits[split]):
        with Image.open(p) as im:
            # igual que la app: ventanas, unión de duplicados y sin recuadros encimados
            out[p] = drop_nested(detector.predict(im.convert("RGB")))
        if i % 100 == 0:
            print(f"  {split}: {i}/{len(splits[split])}")
    return out


def scored(preds: dict) -> list:
    """(puntaje, acierta con cualquier defecto, acierta con el mismo tipo) por recuadro de defecto."""
    rows = []
    for p, boxes in preds.items():
        truth = defects(items[p])
        for b in defects(boxes):
            any_hit = any(iou(b, t) >= IOU_MATCH for t in truth)
            same_hit = any(iou(b, t) >= IOU_MATCH and t.label == b.label for t in truth)
            rows.append((b.score, any_hit, same_hit))
    return sorted(rows, reverse=True)


def curve(rows: list) -> list:
    points, hits, same = [], 0, 0
    total_hits = sum(r[1] for r in rows) or 1
    for n, (score, any_hit, same_hit) in enumerate(rows, start=1):
        hits += any_hit
        same += same_hit
        points.append({"umbral": round(score, 4), "rojos": n, "precision": hits / n,
                       "precision_mismo_tipo": same / n, "aciertos_en_rojo": hits / total_hits})
    return points


def summary(rows: list, threshold: float) -> dict:
    red = [r for r in rows if r[0] >= threshold]
    yellow = [r for r in rows if r[0] < threshold]
    pct = lambda xs, k: round(sum(x[k] for x in xs) / len(xs), 3) if xs else None
    return {"rojos": len(red), "precision_rojos": pct(red, 1), "precision_rojos_mismo_tipo": pct(red, 2),
            "amarillos": len(yellow), "precision_amarillos": pct(yellow, 1),
            "defectos_reales_en_rojo": round(sum(r[1] for r in red) / max(1, sum(r[1] for r in rows)), 3)}


print("Validación…")
val_rows = scored(run("val"))
points = curve(val_rows)
ok = [pt for pt in points if pt["rojos"] >= MIN_RED and pt["precision"] >= TARGET_PRECISION]
# El corte más bajo que sigue cumpliendo (el último en orden de puntaje descendente)
chosen = ok[-1]["umbral"] if ok else 1.0
chosen = max(chosen, UMBRAL_MODELO)
print(f"Umbral alto elegido: {chosen} ({'cumple' if ok else 'ningún corte llega al 80 %: todo amarillo'})")

print("Prueba…")
test_rows = scored(run("test"))
result = {
    "modulo": MODULO,
    "umbral_modelo": UMBRAL_MODELO,
    "umbral_alto": chosen,
    "objetivo_precision": TARGET_PRECISION,
    "validacion": summary(val_rows, chosen),
    "prueba": summary(test_rows, chosen),
    "curva_validacion": points[::max(1, len(points) // 60)],
}
print(json.dumps({k: v for k, v in result.items() if k != "curva_validacion"}, indent=2, ensure_ascii=False))
(WORK / f"calibracion_{MODULO}.json").write_text(json.dumps(result, indent=2, ensure_ascii=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([pt["umbral"] for pt in points], [pt["precision"] for pt in points], label="precisión de los rojos")
ax.plot([pt["umbral"] for pt in points], [pt["aciertos_en_rojo"] for pt in points],
        label="defectos reales que quedan en rojo")
ax.axvline(chosen, color="red", ls="--", label=f"corte elegido {chosen:.2f}")
ax.axhline(TARGET_PRECISION, color="gray", ls=":")
ax.set_xlabel("puntaje del modelo"); ax.set_ylim(0, 1.02); ax.legend(); ax.set_title(f"Calibración · {MODULO}")
plt.tight_layout(); plt.savefig(WORK / f"calibracion_{MODULO}.png", dpi=120); plt.show()

In [ ]:
print("Listo. Pasale a Claude los calibracion_*.json de Output.")